# 🥇 Notebook 03 — Gold Aggregates & ML Feature Store
**Layer**: Gold — analytics-ready, ML-ready, analyst-facing
**Inputs**: `silver.clean_claims`, `silver.hcc_mapped_claims`, `silver.member_profile`
**Outputs**:
- `gold.member_raf_scores` — final RAF score, risk tier, PMPM per member
- `gold.utilization_summary` — IP/ED/PMPM by member × period
- `gold.monthly_trends` — DiD-ready cohort aggregates (3-month rolling avg)
- `ml_features.risk_feature_store` — 33-feature ML matrix (pre-period only, no leakage)
**Runtime**: ~8–12 min


## 1. Setup

In [0]:
import sys, os

# Auto-detect repo root — works for any user / workspace path
_nb_path  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/Workspace" + "/".join(_nb_path.split("/")[:-2])

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

CATALOG       = "medicare_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"
ML_SCHEMA     = "ml_features"
VOLUME_PATH   = f"/Volumes/{CATALOG}/default/landing_zone/raw"

print(f"Repo root  : {repo_root}")
print(f"Catalog    : {CATALOG}")
print(f"Volume path: {VOLUME_PATH}")
_ok = os.path.exists(os.path.join(repo_root, "src"))
print("src/ found : ✅" if _ok else "❌ src/ missing — check repo clone")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{ML_SCHEMA}")
print(f"✅ Schemas ready: {CATALOG}.{GOLD_SCHEMA}, {CATALOG}.{ML_SCHEMA}")


## 2. Member-level RAF scores

**RAF formula** (CMS HCC v28):
```
RAF = demographic_raf
    + Σ(unique HCC RAF coefficients across all member claims)
    + Σ(interaction term RAF: CHF×AFib +0.175, CHF×DM +0.156, CKD×DM +0.156)
```

**Risk tier thresholds** (aligned with ACO care management criteria):
| Tier | RAF | Programme response |
|------|-----|--------------------|
| `high` | ≥ 2.0 | Intensive case management |
| `moderate` | 1.2 – 2.0 | Disease management outreach |
| `low` | < 1.2 | Standard care pathway |


In [0]:
BENCHMARK_PMPM = 9_800.0 / 12  # CMS benchmark ($9,800/member/year)

# Aggregate HCC burden per member across ALL claims (max, not sum — no double-counting)
hcc_per_member = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims")
    .groupBy("bene_id")
    .agg(
        F.max("hcc_burden_count").alias("max_hcc_burden"),
        F.max("hcc_raf_total").alias("max_hcc_raf"),
        F.max("hcc_interaction_total").alias("max_interaction_raf"),
        F.max(F.col("has_chf").cast("int")).alias("has_chf"),
        F.max(F.col("has_afib").cast("int")).alias("has_afib"),
        F.max(F.col("has_diabetes").cast("int")).alias("has_diabetes"),
        F.max(F.col("has_ckd").cast("int")).alias("has_ckd"),
        F.max(F.col("has_cancer").cast("int")).alias("has_cancer"),
        F.max(F.col("has_copd").cast("int")).alias("has_copd"),
        F.max(F.col("has_depression").cast("int")).alias("has_depression"),
        F.max(F.col("has_metastatic").cast("int")).alias("has_metastatic"),
        F.sum("claim_amount").alias("total_claim_amount"),
        F.count("claim_id").alias("total_claims"),
        F.sum(F.col("is_inpatient").cast("int")).alias("ip_admit_count"),
        F.sum(F.col("is_ed_visit").cast("int")).alias("ed_visit_count"),
        # Pre/post split for DiD
        F.sum(F.when(F.col("period") == "pre",  F.col("claim_amount"))).alias("pre_total_cost"),
        F.sum(F.when(F.col("period") == "post", F.col("claim_amount"))).alias("post_total_cost"),
        F.countDistinct(
            F.when(F.col("period") == "pre",
                   F.concat_ws("-", "claim_year", "claim_month"))
        ).alias("pre_months"),
        F.countDistinct(
            F.when(F.col("period") == "post",
                   F.concat_ws("-", "claim_year", "claim_month"))
        ).alias("post_months"),
    )
)

# Join with demographics
members = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.member_profile")
raf_df  = members.join(hcc_per_member, on="bene_id", how="left").fillna(0)

# Final RAF score + derived metrics
raf_scores = (
    raf_df
    .withColumn("raf_score",
        F.col("demographic_raf") +
        F.coalesce(F.col("max_hcc_raf"),        F.lit(0.0)) +
        F.coalesce(F.col("max_interaction_raf"), F.lit(0.0))
    )
    .withColumn("risk_tier",
        F.when(F.col("raf_score") >= 2.0, "high")
         .when(F.col("raf_score") >= 1.2, "moderate")
         .otherwise("low")
    )
    .withColumn("estimated_annual_cost",
        F.col("raf_score") * BENCHMARK_PMPM * 12)
    .withColumn("actual_pmpm",
        F.col("total_claim_amount") /
        F.greatest(F.col("pre_months") + F.col("post_months"), F.lit(1))
    )
    .withColumn("pre_pmpm",
        F.col("pre_total_cost") / F.greatest(F.col("pre_months"), F.lit(1)))
    .withColumn("post_pmpm",
        F.col("post_total_cost") / F.greatest(F.col("post_months"), F.lit(1)))
    .withColumn("_pipeline_layer", F.lit("gold"))
    .withColumn("_ingested_at",    F.current_timestamp())
)

(
    raf_scores.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.member_raf_scores")
)
spark.sql(
    f"OPTIMIZE {CATALOG}.{GOLD_SCHEMA}.member_raf_scores "
    f"ZORDER BY (bene_id, risk_tier)"
)
print(f"✅ {CATALOG}.{GOLD_SCHEMA}.member_raf_scores written")

display(
    raf_scores.groupBy("risk_tier").agg(
        F.count("*").alias("n_members"),
        F.round(F.avg("raf_score"), 3).alias("mean_raf"),
        F.round(F.min("raf_score"), 3).alias("min_raf"),
        F.round(F.max("raf_score"), 3).alias("max_raf"),
        F.round(F.avg("estimated_annual_cost"), 0).alias("mean_est_cost_usd"),
        F.round(F.avg("actual_pmpm"), 2).alias("mean_actual_pmpm"),
    ).orderBy(F.desc("mean_raf"))
)


## 3. Utilization summary — IP/ED/PMPM by member × period

In [0]:
util = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.clean_claims")
    .filter("_quality_pass = true")
    .groupBy("bene_id", "period", "intervention_arm")
    .agg(
        F.count("claim_id").alias("n_claims"),
        F.sum("claim_amount").alias("total_cost"),
        F.sum("is_inpatient").alias("ip_admits"),
        F.sum("is_ed_visit").alias("ed_visits"),
        F.sum("is_specialist").alias("specialist_visits"),
        F.sum("is_primary_care").alias("primary_visits"),
        F.countDistinct(
            F.concat_ws("-", "claim_year", "claim_month")
        ).alias("n_months"),
    )
    .withColumn("pmpm_cost",
        F.col("total_cost") / F.greatest(F.col("n_months"), F.lit(1)))
    .withColumn("ip_rate_per_1000",
        F.col("ip_admits") * 1000 /
        F.greatest(F.col("n_months"), F.lit(1)) / 12)
    .withColumn("ed_rate_per_1000",
        F.col("ed_visits") * 1000 /
        F.greatest(F.col("n_months"), F.lit(1)) / 12)
    .withColumn("_pipeline_layer", F.lit("gold"))
)

(
    util.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .partitionBy("period")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.utilization_summary")
)
print(f"✅ {CATALOG}.{GOLD_SCHEMA}.utilization_summary written")

display(
    util.groupBy("period", "intervention_arm").agg(
        F.countDistinct("bene_id").alias("n_members"),
        F.round(F.avg("pmpm_cost"), 2).alias("mean_pmpm"),
        F.round(F.avg("ip_rate_per_1000"), 1).alias("ip_rate_per_1000"),
        F.round(F.avg("ed_rate_per_1000"), 1).alias("ed_rate_per_1000"),
    ).orderBy(F.desc("period"), "intervention_arm")
)


## 4. Monthly cohort trends — DiD-ready

One row per `(year, month, intervention_arm)`. The 3-month rolling average uses a
Spark Window function on the ordered cohort series. This table requires **no reshaping**
to run a DiD estimator — it is structured as the regression input from the moment it lands.


In [0]:
trends = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.clean_claims")
    .filter("_quality_pass = true")
    .groupBy("claim_year", "claim_month", "intervention_arm")
    .agg(
        F.countDistinct("bene_id").alias("n_members"),
        F.sum("claim_amount").alias("total_cost"),
        F.sum("is_inpatient").alias("ip_admits"),
        F.sum("is_ed_visit").alias("ed_visits"),
    )
    .withColumn("pmpm_cost",
        F.col("total_cost") / F.greatest(F.col("n_members"), F.lit(1)))
    .withColumn("ip_rate_per_1000",
        F.col("ip_admits") * 1000 / F.greatest(F.col("n_members"), F.lit(1)))
    .withColumn("ed_rate_per_1000",
        F.col("ed_visits") * 1000 / F.greatest(F.col("n_members"), F.lit(1)))
    .withColumn("period",
        F.when(F.col("claim_year") >= 2023, "post").otherwise("pre"))
    # 3-month rolling avg via Spark Window
    .withColumn("pmpm_rolling_3m",
        F.avg("pmpm_cost").over(
            Window.partitionBy("intervention_arm")
                  .orderBy("claim_year", "claim_month")
                  .rowsBetween(-2, 0)
        )
    )
    .withColumn("_pipeline_layer", F.lit("gold"))
    .orderBy("claim_year", "claim_month", "intervention_arm")
)

(
    trends.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.monthly_trends")
)
print(f"✅ {CATALOG}.{GOLD_SCHEMA}.monthly_trends written")

# Quick DiD estimate from the table
rows = trends.groupBy("period", "intervention_arm").agg(
    F.mean("pmpm_cost").alias("mean_pmpm")
).collect()
did = {(r["period"], r["intervention_arm"]): r["mean_pmpm"] for r in rows}
if len(did) == 4:
    att = (did[("post",1)]-did[("pre",1)]) - (did[("post",0)]-did[("pre",0)])
    print(f"\nDiD ATT estimate: ${att:,.2f}/member/month = ${att*12:,.0f}/member/year")


## 5. ML feature store — 33 features, zero leakage

**33 feature groups**:
- Demographic (5): age, sex_male, dual_eligible, demographic_raf, age_bracket
- HCC burden (3): max_hcc_burden, max_hcc_raf, max_interaction_raf
- Condition flags (8): has_chf, has_afib, has_diabetes, has_ckd, has_cancer, has_copd, has_depression, has_metastatic
- Pre-period utilization (11): pre_ip_admits, pre_ed_visits, pre_specialist_visits, pre_primary_visits, pre_n_months, pre_total_cost, pre_avg_claim, pre_max_claim, pre_pmpm, pre_ip_rate, pre_ed_rate
- RAF & cost (3): raf_score, estimated_annual_cost, actual_pmpm
- Log-transformed (3): log_pre_pmpm, log_estimated_cost, log_pre_total_cost

**Leakage prevention**: Only pre-period utilization features. Post-period data belongs
in outcome measurement, not the feature matrix for prospective risk prediction.


In [0]:
# Pre-period utilization — no post-period data (no leakage)
pre_util = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.clean_claims")
    .filter("_quality_pass = true AND period = 'pre'")
    .groupBy("bene_id")
    .agg(
        F.sum("is_inpatient").alias("pre_ip_admits"),
        F.sum("is_ed_visit").alias("pre_ed_visits"),
        F.sum("is_specialist").alias("pre_specialist_visits"),
        F.sum("is_primary_care").alias("pre_primary_visits"),
        F.countDistinct(
            F.concat_ws("-", "claim_year", "claim_month")
        ).alias("pre_n_months"),
        F.sum("claim_amount").alias("pre_total_cost"),
        F.avg("claim_amount").alias("pre_avg_claim"),
        F.max("claim_amount").alias("pre_max_claim"),
    )
    .withColumn("pre_pmpm",
        F.col("pre_total_cost") / F.greatest(F.col("pre_n_months"), F.lit(1)))
    .withColumn("pre_ip_rate",
        F.col("pre_ip_admits") / F.greatest(F.col("pre_n_months"), F.lit(1)))
    .withColumn("pre_ed_rate",
        F.col("pre_ed_visits") / F.greatest(F.col("pre_n_months"), F.lit(1)))
)

# Join all feature groups
feat_store = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.member_raf_scores")
    .join(pre_util, on="bene_id", how="left")
    .fillna(0)
    # Boolean flags → int (clean numeric matrix for XGBoost)
    .withColumn("sex_male", (F.col("sex") == "M").cast("int"))
    # Log-transform skewed cost features
    .withColumn("log_pre_pmpm",       F.log1p("pre_pmpm"))
    .withColumn("log_estimated_cost", F.log1p("estimated_annual_cost"))
    .withColumn("log_pre_total_cost", F.log1p("pre_total_cost"))
    .select(
        "bene_id",
        # Demographic (5)
        "age", "sex_male", "dual_eligible", "demographic_raf", "age_bracket",
        # HCC burden (3)
        "max_hcc_burden", "max_hcc_raf", "max_interaction_raf",
        # Condition flags (8)
        "has_chf", "has_afib", "has_diabetes", "has_ckd",
        "has_cancer", "has_copd", "has_depression", "has_metastatic",
        # Pre-period utilization (11)
        "pre_ip_admits", "pre_ed_visits", "pre_specialist_visits",
        "pre_primary_visits", "pre_n_months", "pre_total_cost",
        "pre_avg_claim", "pre_max_claim", "pre_pmpm",
        "pre_ip_rate", "pre_ed_rate",
        # RAF & cost (3)
        "raf_score", "estimated_annual_cost", "actual_pmpm",
        # Log-transformed (3)
        "log_pre_pmpm", "log_estimated_cost", "log_pre_total_cost",
        # Labels
        "risk_tier", "intervention_arm",
    )
    .withColumn("_pipeline_layer", F.lit("ml_features"))
)

(
    feat_store.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{ML_SCHEMA}.risk_feature_store")
)
spark.sql(
    f"OPTIMIZE {CATALOG}.{ML_SCHEMA}.risk_feature_store ZORDER BY (risk_tier)"
)

n = feat_store.count()
n_feat = 33  # validated against CLASSIFICATION_FEATURES in risk_model.py
print(f"✅ {CATALOG}.{ML_SCHEMA}.risk_feature_store — {n:,} members, {n_feat} features")
display(
    feat_store.groupBy("risk_tier").agg(
        F.count("*").alias("n"),
        F.round(F.avg("raf_score"), 3).alias("mean_raf"),
        F.round(F.avg("pre_pmpm"), 2).alias("mean_pre_pmpm"),
        F.round(F.avg("max_hcc_burden"), 1).alias("mean_hcc_burden"),
        F.round(F.avg("log_pre_pmpm"), 3).alias("mean_log_pre_pmpm"),
    ).orderBy(F.desc("mean_raf"))
)


## 6. MSSP Shared Savings Projection

In [0]:
from src.gold.shared_savings import SharedSavingsCalculator

calc = SharedSavingsCalculator()

# From Gold actuals
stats = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.member_raf_scores")
    .agg(
        F.mean("actual_pmpm").alias("mean_pmpm"),
        F.count("*").alias("n_members")
    ).collect()[0]
)
result_actual = calc.compute(
    benchmark_pmpm = 9800 / 12,
    actual_pmpm    = float(stats["mean_pmpm"]),
    n_lives        = int(stats["n_members"])
)
print("=== From Gold actuals ===")
print(result_actual)

# Scale projection using ATT from companion RAF pipeline
print("\n=== Scale projection — ATT = $391/member/year ===")
display(spark.createDataFrame(calc.project(att_pmpm=391.0).reset_index()))


## ✅ Gold complete

| Table | Purpose |
|-------|---------|
| `gold.member_raf_scores` | Final RAF, risk tier, PMPM per member |
| `gold.utilization_summary` | IP/ED/PMPM by member × period |
| `gold.monthly_trends` | DiD-ready cohort × month × arm |
| `ml_features.risk_feature_store` | 33-feature XGBoost input |

**Next**: Run `04_mlflow_model` →
